# TestingAI - Оценка нейронных сетей для дефектоскопии труб

## Этапы:
1. Загрузка данных из .raw файлов
2. Стратифицированное разделение на train/val/test (70%/15%/15%)
3. Стандартизация признаков (StandardScaler, обученный только на train)
4. Аугментация данных (гауссовский шум, временной сдвиг, масштабирование амплитуды)
5. Сохранение индексов тестовой выборки для воспроизводимости
6. Оценка моделей: 1D_CNN, CNN_LSTM, 1D_CNN_XGBoost, PipeTransformer

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score, 
    precision_recall_fscore_support,
    roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import struct
import pickle
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## 1. Функции загрузки данных

In [ ]:
def read_raw_file(filepath):
    """Читает .raw файл и возвращает данные в формате для нейросети"""
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"❌ Файл не найден: {path}")

    with open(path, 'rb') as f:
        data = f.read()

    if len(data) < 17:
        raise ValueError("⚠️ Файл слишком короткий")

    # Парсинг заголовка (Big-Endian)
    x = struct.unpack_from('>i', data, 0)[0]  # длина
    y = struct.unpack_from('>i', data, 4)[0]  # ширина (8)
    thicknom = struct.unpack_from('>d', data, 8)[0]
    defective = struct.unpack_from('>B', data, 16)[0]

    # Извлечение матрицы
    num_values = x * y
    matrix = struct.unpack(f'>{num_values}d', data[17:17+num_values*8])
    
    # Преобразование в numpy массив и ресейп в (length, 8)
    matrix_np = np.array(matrix, dtype=np.float32).reshape(x, y)
    
    return {
        'matrix': matrix_np,        # Форма: (length, 8)
        'defective': defective,     # 0 или 1
        'thicknom': thicknom,
        'filename': path.name
    }


def load_dataset_from_folder(folder_path, verbose=True):
    """Загружает все .raw файлы из папки"""
    folder = Path(folder_path)
    raw_files = list(folder.glob('*.raw'))
    
    if not raw_files:
        raise FileNotFoundError(f"❌ Не найдено .raw файлов в {folder_path}")
    
    if verbose:
        print(f"📁 Найдено файлов: {len(raw_files)}")
    
    data_list = []
    labels = []
    filenames = []
    thicknoms = []
    
    good_count = 0
    bad_count = 0
    
    for file_path in raw_files:
        try:
            result = read_raw_file(file_path)
            data_list.append(result['matrix'])
            labels.append(result['defective'])
            filenames.append(result['filename'])
            thicknoms.append(result['thicknom'])
            
            if result['defective'] == 0:
                good_count += 1
            else:
                bad_count += 1
                
            if verbose and len(data_list) % 10 == 0:
                print(f"  Загружено: {len(data_list)}/{len(raw_files)}")
                
        except Exception as e:
            print(f"⚠️ Ошибка при чтении {file_path.name}: {e}")
            continue
    
    if verbose:
        print(f"\n✅ Загружено: {len(data_list)} файлов")
        print(f"   🟢 Годных (0): {good_count}")
        print(f"   🔴 Бракованных (1): {bad_count}")
        print(f"   📊 Размер матрицы: {data_list[0].shape}")
    
    return data_list, np.array(labels), np.array(filenames), np.array(thicknoms)

## 2. Аугментация данных

In [ ]:
def augment_data(data_list, labels, n_augmented_per_sample=1, random_state=42):
    """
    Применяет аугментацию к данным для повышения устойчивости модели.
    
    Параметры:
    - data_list: список numpy массивов формы (length, 8)
    - labels: список меток классов
    - n_augmented_per_sample: количество аугментированных копий на образец
    - random_state: seed для воспроизводимости
    
    Виды аугментации:
    1. Гауссовский шум: σ ∈ [0.01, 0.05]
    2. Временной сдвиг: ±3 отсчета
    3. Масштабирование амплитуды: ×[0.9, 1.1]
    """
    np.random.seed(random_state)
    
    augmented_data = []
    augmented_labels = []
    
    for data, label in zip(data_list, labels):
        # Оригинал
        augmented_data.append(data.copy())
        augmented_labels.append(label)
        
        # Создаем n_augmented_per_sample аугментированных копий
        for _ in range(n_augmented_per_sample):
            augmented = data.copy()
            
            # 1. Гауссовский шум σ ∈ [0.01, 0.05]
            sigma = np.random.uniform(0.01, 0.05)
            noise = np.random.normal(0, sigma, augmented.shape)
            augmented = augmented + noise
            
            # 2. Временной сдвиг ±3 отсчета
            shift = np.random.randint(-3, 4)  # от -3 до +3 включительно
            if shift != 0:
                shifted = np.zeros_like(augmented)
                if shift > 0:
                    shifted[shift:, :] = augmented[:-shift, :]
                    shifted[:shift, :] = augmented[0, :]  # заполняем начало первым значением
                else:
                    shifted[:shift, :] = augmented[-shift:, :]
                    shifted[shift:, :] = augmented[-1, :]  # заполняем конец последним значением
                augmented = shifted
            
            # 3. Масштабирование амплитуды ×[0.9, 1.1]
            scale_factor = np.random.uniform(0.9, 1.1)
            augmented = augmented * scale_factor
            
            augmented_data.append(augmented)
            augmented_labels.append(label)
    
    print(f"🔄 Аугментация завершена:")
    print(f"   Исходно: {len(data_list)} образцов")
    print(f"   После аугментации: {len(augmented_data)} образцов")
    print(f"   Класс 0: {sum(np.array(augmented_labels)==0)}, Класс 1: {sum(np.array(augmented_labels)==1)}")
    
    return augmented_data, np.array(augmented_labels)

## 3. Подготовка данных и разделение выборок

In [ ]:
def prepare_and_split_data(data_list, labels, filenames, thicknoms,
                           train_size=0.7, val_size=0.15, test_size=0.15,
                           apply_augmentation=False, n_augmented=1,
                           random_state=42, save_indices=True, output_dir='./test_indices'):
    """
    Полный цикл подготовки данных:
    1. Опциональная аугментация
    2. Стратифицированное разделение на train/val/test
    3. Стандартизация признаков (StandardScaler, обученный только на train)
    4. Сохранение индексов тестовой выборки
    
    Параметры:
    - data_list: список numpy массивов формы (length, 8)
    - labels: массив меток классов
    - filenames: массив имен файлов
    - thicknoms: массив значений thicknom
    - train_size, val_size, test_size: доли разбиения
    - apply_augmentation: применять ли аугментацию к обучающей выборке
    - n_augmented: количество аугментированных копий на образец
    - random_state: seed для воспроизводимости
    - save_indices: сохранять ли индексы тестовой выборки
    - output_dir: директория для сохранения индексов
    """
    print("="*60)
    print("🔄 ПОДГОТОВКА ДАННЫХ И РАЗДЕЛЕНИЕ ВЫБОРОК")
    print("="*60)
    
    # Сначала разделим данные на train+val и test
    # Используем stratify для сохранения баланса классов
    temp_size = val_size / (train_size + val_size)  # доля val от (train+val)
    
    # Первое разделение: (train+val) vs test
    X_trainval, X_test, y_trainval, y_test, idx_trainval, idx_test = train_test_split(
        data_list, labels, np.arange(len(data_list)),
        test_size=test_size,
        random_state=random_state,
        stratify=labels
    )
    
    # Второе разделение: train vs val
    X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
        X_trainval, y_trainval, idx_trainval,
        test_size=temp_size,
        random_state=random_state,
        stratify=y_trainval
    )
    
    print(f"\n📊 РАЗМЕРЫ ВЫБОРОК ДО АУГМЕНТАЦИИ:")
    print(f"   Train: {len(X_train)} образцов ({100*len(X_train)/len(data_list):.1f}%)")
    print(f"   Val:   {len(X_val)} образцов ({100*len(X_val)/len(data_list):.1f}%)")
    print(f"   Test:  {len(X_test)} образцов ({100*len(X_test)/len(data_list):.1f}%)")
    
    # Применяем аугментацию только к обучающей выборке
    if apply_augmentation:
        print(f"\n🔄 ПРИМЕНЕНИЕ АУГМЕНТАЦИИ К TRAIN ВЫБОРКЕ...")
        X_train, y_train = augment_data(X_train, y_train, n_augmented_per_sample=n_augmented, random_state=random_state)
    
    # Стандартизация признаков
    print(f"\n📏 СТАНДАРТИЗАЦИЯ ПРИЗНАКОВ (StandardScaler)...")
    
    # Объединяем все данные train для обучения скалера
    all_train_data = np.vstack(X_train)
    scaler = StandardScaler().fit(all_train_data)
    
    # Применяем стандартизацию ко всем выборкам
    X_train_scaled = [scaler.transform(sample) for sample in X_train]
    X_val_scaled = [scaler.transform(sample) for sample in X_val]
    X_test_scaled = [scaler.transform(sample) for sample in X_test]
    
    print(f"   ✅ Скалер обучен на {len(X_train)} train образцах")
    print(f"   ✅ Применен ко всем выборкам")
    
    # Получаем соответствующие filenames и thicknoms
    filenames_train = filenames[idx_train]
    filenames_val = filenames[idx_val]
    filenames_test = filenames[idx_test]
    
    thicknoms_train = thicknoms[idx_train]
    thicknoms_val = thicknoms[idx_val]
    thicknoms_test = thicknoms[idx_test]
    
    # Проверка баланса классов
    print(f"\n📊 БАЛАНС КЛАССОВ В ВЫБОРКАХ:")
    for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
        class_0 = sum(y==0)
        class_1 = sum(y==1)
        total = len(y)
        print(f"   {name}: Класс 0={class_0} ({100*class_0/total:.1f}%), Класс 1={class_1} ({100*class_1/total:.1f}%)")
    
    # Сохранение индексов тестовой выборки
    if save_indices:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        indices_path = Path(output_dir) / f'test_indices_{timestamp}.pkl'
        
        test_info = {
            'indices': idx_test.tolist(),
            'filenames': filenames_test.tolist(),
            'labels': y_test.tolist(),
            'thicknoms': thicknoms_test.tolist(),
            'random_state': random_state,
            'split_sizes': {'train': train_size, 'val': val_size, 'test': test_size},
            'augmentation_applied': apply_augmentation,
            'n_augmented': n_augmented if apply_augmentation else 0
        }
        
        with open(indices_path, 'wb') as f:
            pickle.dump(test_info, f)
        
        print(f"\n💾 Индексы тестовой выборки сохранены: {indices_path}")
    
    print("\n" + "="*60)
    print("✅ ДАННЫЕ ПОДГОТОВЛЕНЫ!")
    print("="*60)
    
    return {
        'X_train': X_train_scaled,
        'X_val': X_val_scaled,
        'X_test': X_test_scaled,
        'y_train': y_train,
        'y_val': y_val,
        'y_test': y_test,
        'filenames_train': filenames_train,
        'filenames_val': filenames_val,
        'filenames_test': filenames_test,
        'thicknoms_train': thicknoms_train,
        'thicknoms_val': thicknoms_val,
        'thicknoms_test': thicknoms_test,
        'scaler': scaler,
        'test_indices': idx_test
    }

## 4. Dataset и DataLoader для PyTorch

In [ ]:
class PipeDataset(Dataset):
    """
    Dataset для работы с данными толщинометрии труб.
    """
    def __init__(self, data_list, labels):
        """
        data_list: список numpy массивов формы (length, 8)
        labels: список меток классов (int)
        """
        self.data_list = data_list
        self.labels = labels
        
        # Преобразуем в тензоры
        self.data_list = [
            torch.FloatTensor(sample) 
            for sample in self.data_list
        ]
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.data_list)
    
    def __getitem__(self, idx):
        return self.data_list[idx], self.labels[idx]


def collate_fn(batch):
    """
    Формирует батч из данных переменной длины.
    Дополняет (pad) короткие последовательности нулями до самой длинной в батче.
    """
    data_list, labels = zip(*batch)
    
    # Находим максимальную длину в текущем батче
    max_len = max([d.shape[0] for d in data_list])
    
    # Создаем тензоры с паддингом
    padded_data = []
    lengths = []
    
    for data in data_list:
        length = data.shape[0]
        lengths.append(length)
        
        # Дополняем нулями до max_len: (length, 8) -> (max_len, 8)
        padding = torch.zeros(max_len - length, data.shape[1])
        padded = torch.cat([data, padding], dim=0)
        padded_data.append(padded)
    
    # Стек: (Batch, Max_Len, 8)
    data_batch = torch.stack(padded_data)
    labels_batch = torch.stack(labels)
    
    # Создаем маску для CNN (чтобы игнорировать паддинг при пулинге)
    mask = torch.zeros(data_batch.shape[:2])
    for i, length in enumerate(lengths):
        mask[i, :length] = 1
    
    return data_batch, labels_batch, mask, torch.tensor(lengths)


def create_dataloaders(data_dict, batch_size=32, num_workers=0):
    """
    Создает DataLoader для train, val и test выборок.
    """
    train_dataset = PipeDataset(data_dict['X_train'], data_dict['y_train'])
    val_dataset = PipeDataset(data_dict['X_val'], data_dict['y_val'])
    test_dataset = PipeDataset(data_dict['X_test'], data_dict['y_test'])
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        collate_fn=collate_fn,
        num_workers=num_workers
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        collate_fn=collate_fn,
        num_workers=num_workers
    )
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        collate_fn=collate_fn,
        num_workers=num_workers
    )
    
    print(f"\n📦 DataLoader созданы:")
    print(f"   Train batches: {len(train_loader)}")
    print(f"   Val batches: {len(val_loader)}")
    print(f"   Test batches: {len(test_loader)}")
    
    return train_loader, val_loader, test_loader

## 5. Функции оценки моделей

In [ ]:
def evaluate_model(model, dataloader, device='cpu'):
    """
    Оценивает модель на даталоадере.
    Возвращает метрики и предсказания.
    """
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for data, labels, mask, lengths in dataloader:
            data, labels = data.to(device), labels.to(device)
            
            # Предсказание
            outputs = model(data)
            
            # Получаем вероятности и классы
            if isinstance(outputs, tuple):
                outputs = outputs[0]  # Если модель возвращает кортеж
            
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(probs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())  # Вероятность класса 1
            all_labels.extend(labels.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    # Метрики
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary', zero_division=0)
    
    try:
        auc_roc = roc_auc_score(all_labels, all_probs)
    except:
        auc_roc = 0.5
    
    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc_roc': auc_roc
    }
    
    return metrics, all_preds, all_probs, all_labels


def print_classification_report(y_true, y_pred, target_names=None):
    """Выводит подробный отчет о классификации"""
    print("\n" + "="*60)
    print("📋 ОТЧЕТ О КЛАССИФИКАЦИИ")
    print("="*60)
    print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))


def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix', save_path=None):
    """Построение матрицы ошибок"""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Class 0', 'Class 1'],
                yticklabels=['Class 0', 'Class 1'])
    plt.title(title, fontsize=14)
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('True', fontsize=12)
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 Матрица ошибок сохранена: {save_path}")
    else:
        plt.show()
    
    plt.close()
    return cm


def plot_roc_curve(y_true, y_probs, title='ROC Curve', save_path=None):
    """Построение ROC-кривой"""
    from sklearn.metrics import roc_curve
    
    fpr, tpr, thresholds = roc_curve(y_true, y_probs)
    auc = roc_auc_score(y_true, y_probs)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(loc="lower right")
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 ROC-кривая сохранена: {save_path}")
    else:
        plt.show()
    
    plt.close()
    return fpr, tpr, auc

## 6. Загрузчик моделей

In [ ]:
def load_1d_cnn_model(model_path, input_channels=8, n_classes=2, device='cpu'):
    """
    Загружает модель 1D_CNN из файла.
    """
    # Импортируем архитектуру из 1D_CNN.ipynb
    import importlib.util
    spec = importlib.util.spec_from_loader(
        'cnn_module',
        loader=None
    )
    
    # Определяем архитектуру локально
    class Pipe1DCNN(nn.Module):
        def __init__(self, input_channels=8, n_classes=2, dropout=0.3):
            super().__init__()
            self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=3, padding=1)
            self.bn1 = nn.BatchNorm1d(64)
            self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
            self.bn2 = nn.BatchNorm1d(128)
            self.conv3 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
            self.bn3 = nn.BatchNorm1d(256)
            self.pool = nn.AdaptiveMaxPool1d(1)
            self.dropout = nn.Dropout(dropout)
            self.classifier = nn.Sequential(
                nn.Linear(256, 64),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(64, n_classes)
            )
        
        def forward(self, x):
            # x: (Batch, Length, Channels) -> (Batch, Channels, Length)
            x = x.transpose(1, 2)
            x = self.dropout(torch.relu(self.bn1(self.conv1(x))))
            x = self.dropout(torch.relu(self.bn2(self.conv2(x))))
            x = self.dropout(torch.relu(self.bn3(self.conv3(x))))
            x = self.pool(x).squeeze(-1)
            x = self.classifier(x)
            return x
    
    model = Pipe1DCNN(input_channels=input_channels, n_classes=n_classes).to(device)
    
    if model_path and Path(model_path).exists():
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict):
            model.load_state_dict(checkpoint.get('model_state_dict', checkpoint))
        else:
            model.load_state_dict(checkpoint)
        print(f"✅ Модель загружена из: {model_path}")
    else:
        print(f"⚠️ Файл модели не найден: {model_path}")
    
    return model


def load_cnn_lstm_model(model_path, input_channels=8, n_classes=2, hidden_size=128, device='cpu'):
    """
    Загружает модель CNN_LSTM из файла.
    """
    class CNNLSTM(nn.Module):
        def __init__(self, input_channels=8, n_classes=2, hidden_size=128, num_layers=2):
            super().__init__()
            self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=3, padding=1)
            self.bn1 = nn.BatchNorm1d(64)
            self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
            self.bn2 = nn.BatchNorm1d(128)
            self.pool = nn.MaxPool1d(2)
            self.lstm = nn.LSTM(128, hidden_size, num_layers=num_layers, 
                               batch_first=True, bidirectional=True, dropout=0.3)
            self.classifier = nn.Sequential(
                nn.Linear(hidden_size * 2, 64),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, n_classes)
            )
        
        def forward(self, x):
            x = x.transpose(1, 2)  # (B, C, L)
            x = self.pool(torch.relu(self.bn1(self.conv1(x))))
            x = self.pool(torch.relu(self.bn2(self.conv2(x))))  # (B, C, L')
            x = x.transpose(1, 2)  # (B, L', C)
            lstm_out, _ = self.lstm(x)
            out = lstm_out[:, -1, :]  # Последний выход
            return self.classifier(out)
    
    model = CNNLSTM(input_channels=input_channels, n_classes=n_classes, hidden_size=hidden_size).to(device)
    
    if model_path and Path(model_path).exists():
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict):
            model.load_state_dict(checkpoint.get('model_state_dict', checkpoint))
        else:
            model.load_state_dict(checkpoint)
        print(f"✅ Модель загружена из: {model_path}")
    else:
        print(f"⚠️ Файл модели не найден: {model_path}")
    
    return model


def load_transformer_model(model_path, input_channels=8, n_classes=2, device='cpu'):
    """
    Загружает модель PipeTransformer из файла.
    """
    class PositionalEncoding(nn.Module):
        def __init__(self, d_model, max_len=500):
            super().__init__()
            pe = torch.zeros(max_len, d_model)
            position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            self.register_buffer('pe', pe.unsqueeze(0))
        
        def forward(self, x):
            return x + self.pe[:, :x.size(1), :]
    
    class PipeTransformer(nn.Module):
        def __init__(self, input_channels=8, d_model=64, nhead=4, num_layers=3, n_classes=2, dropout=0.1):
            super().__init__()
            self.input_proj = nn.Linear(input_channels, d_model)
            self.pos_encoder = PositionalEncoding(d_model)
            encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, 
                                                       dim_feedforward=d_model*4, dropout=dropout)
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            self.classifier = nn.Sequential(
                nn.Linear(d_model, 32),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(32, n_classes)
            )
        
        def forward(self, x):
            x = self.input_proj(x)  # (B, L, d_model)
            x = self.pos_encoder(x)
            x = x.transpose(0, 1)  # (L, B, d_model)
            x = self.transformer(x)
            x = x.mean(dim=0)  # Global average pooling
            return self.classifier(x)
    
    model = PipeTransformer(input_channels=input_channels, n_classes=n_classes).to(device)
    
    if model_path and Path(model_path).exists():
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict):
            model.load_state_dict(checkpoint.get('model_state_dict', checkpoint))
        else:
            model.load_state_dict(checkpoint)
        print(f"✅ Модель загружена из: {model_path}")
    else:
        print(f"⚠️ Файл модели не найден: {model_path}")
    
    return model


def load_xgboost_model(model_path):
    """
    Загружает модель XGBoost из файла.
    """
    import xgboost as xgb
    
    if model_path and Path(model_path).exists():
        model = xgb.XGBClassifier()
        model.load_model(model_path)
        print(f"✅ XGBoost модель загружена из: {model_path}")
        return model
    else:
        print(f"⚠️ Файл модели не найден: {model_path}")
        return None

## 7. Основная функция тестирования

In [ ]:
def run_full_evaluation(dataset_path='dataset', 
                        models_config=None,
                        batch_size=32,
                        apply_augmentation=False,
                        n_augmented=1,
                        random_state=42,
                        save_results=True,
                        output_dir='./evaluation_results'):
    """
    Запускает полную оценку всех моделей.
    
    Параметры:
    - dataset_path: путь к папке с .raw файлами
    - models_config: словарь с конфигурацией моделей
    - batch_size: размер батча
    - apply_augmentation: применять ли аугментацию
    - n_augmented: количество аугментированных копий
    - random_state: seed для воспроизводимости
    - save_results: сохранять ли результаты
    - output_dir: директория для сохранения результатов
    """
    print("\n" + "#"*60)
    print("# ЗАПУСК ПОЛНОЙ ОЦЕНКИ МОДЕЛЕЙ")
    print("#"*60)
    
    # Определение устройства
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n🖥️ Устройство: {device}")
    
    # Загрузка данных
    print("\n" + "="*60)
    print("📂 ЗАГРУЗКА ДАННЫХ")
    print("="*60)
    data_list, labels, filenames, thicknoms = load_dataset_from_folder(dataset_path)
    
    # Подготовка и разделение данных
    data_dict = prepare_and_split_data(
        data_list, labels, filenames, thicknoms,
        train_size=0.7, val_size=0.15, test_size=0.15,
        apply_augmentation=apply_augmentation,
        n_augmented=n_augmented,
        random_state=random_state,
        save_indices=True,
        output_dir=output_dir
    )
    
    # Создание DataLoader
    train_loader, val_loader, test_loader = create_dataloaders(data_dict, batch_size=batch_size)
    
    # Конфигурация моделей по умолчанию
    if models_config is None:
        models_config = {
            '1D_CNN': {
                'type': 'pytorch',
                'loader': load_1d_cnn_model,
                'model_path': None,  # Укажите путь к сохраненной модели
                'input_channels': 8,
                'n_classes': 2
            },
            'CNN_LSTM': {
                'type': 'pytorch',
                'loader': load_cnn_lstm_model,
                'model_path': None,
                'input_channels': 8,
                'n_classes': 2,
                'hidden_size': 128
            },
            'PipeTransformer': {
                'type': 'pytorch',
                'loader': load_transformer_model,
                'model_path': None,
                'input_channels': 8,
                'n_classes': 2
            },
            '1D_CNN_XGBoost': {
                'type': 'xgboost',
                'loader': load_xgboost_model,
                'model_path': None  # Укажите путь к сохраненной модели .json или .ubj
            }
        }
    
    # Результаты оценки
    all_results = {}
    
    # Оценка каждой модели
    for model_name, config in models_config.items():
        print("\n" + "#"*60)
        print(f"# ОЦЕНКА МОДЕЛИ: {model_name}")
        print("#"*60)
        
        try:
            # Загрузка модели
            if config['type'] == 'pytorch':
                kwargs = {
                    'model_path': config.get('model_path'),
                    'input_channels': config.get('input_channels', 8),
                    'n_classes': config.get('n_classes', 2),
                    'device': device
                }
                if 'hidden_size' in config:
                    kwargs['hidden_size'] = config['hidden_size']
                model = config['loader'](**kwargs)
                model.eval()
                
                # Оценка
                metrics, preds, probs, true_labels = evaluate_model(model, test_loader, device)
                
            elif config['type'] == 'xgboost':
                # Для XGBoost нужно подготовить плоские признаки
                X_test_flat = np.array([sample.flatten() for sample in data_dict['X_test']])
                y_test = data_dict['y_test']
                
                model = config['loader'](config.get('model_path'))
                
                if model is not None:
                    preds = model.predict(X_test_flat)
                    probs = model.predict_proba(X_test_flat)[:, 1]
                    
                    accuracy = accuracy_score(y_test, preds)
                    precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary', zero_division=0)
                    auc_roc = roc_auc_score(y_test, probs)
                    
                    metrics = {
                        'accuracy': accuracy,
                        'precision': precision,
                        'recall': recall,
                        'f1': f1,
                        'auc_roc': auc_roc
                    }
                    true_labels = y_test
                else:
                    print(f"⚠️ Пропуск оценки {model_name} - модель не загружена")
                    continue
            
            # Вывод результатов
            print(f"\n📊 РЕЗУЛЬТАТЫ ДЛЯ {model_name}:")
            print(f"   Accuracy:  {metrics['accuracy']:.4f}")
            print(f"   Precision: {metrics['precision']:.4f}")
            print(f"   Recall:    {metrics['recall']:.4f}")
            print(f"   F1-Score:  {metrics['f1']:.4f}")
            print(f"   AUC-ROC:   {metrics['auc_roc']:.4f}")
            
            # Отчет о классификации
            print_classification_report(true_labels, preds, target_names=['Good', 'Defective'])
            
            # Сохранение результатов
            if save_results:
                Path(output_dir).mkdir(parents=True, exist_ok=True)
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                
                # Сохранение метрик
                results_path = Path(output_dir) / f'{model_name}_metrics_{timestamp}.json'
                with open(results_path, 'w') as f:
                    json.dump({
                        'model_name': model_name,
                        'metrics': metrics,
                        'timestamp': timestamp,
                        'test_size': len(true_labels),
                        'augmentation': apply_augmentation,
                        'random_state': random_state
                    }, f, indent=2)
                
                # Сохранение матрицы ошибок
                cm_path = Path(output_dir) / f'{model_name}_confusion_matrix_{timestamp}.png'
                plot_confusion_matrix(true_labels, preds, 
                                     title=f'{model_name} - Confusion Matrix',
                                     save_path=cm_path)
                
                # Сохранение ROC-кривой
                roc_path = Path(output_dir) / f'{model_name}_roc_curve_{timestamp}.png'
                plot_roc_curve(true_labels, probs,
                              title=f'{model_name} - ROC Curve',
                              save_path=roc_path)
            
            all_results[model_name] = {
                'metrics': metrics,
                'predictions': preds.tolist(),
                'probabilities': probs.tolist(),
                'true_labels': true_labels.tolist()
            }
            
        except Exception as e:
            print(f"❌ Ошибка при оценке {model_name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # Сводная таблица результатов
    print("\n" + "="*60)
    print("📋 СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
    print("="*60)
    
    results_df = pd.DataFrame({
        name: res['metrics'] for name, res in all_results.items()
    })
    print(results_df.T.to_string())
    
    # Сохранение сводных результатов
    if save_results and all_results:
        summary_path = Path(output_dir) / f'all_results_summary_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
        with open(summary_path, 'w') as f:
            json.dump(all_results, f, indent=2)
        print(f"\n💾 Все результаты сохранены в: {output_dir}")
    
    return all_results, data_dict

## 8. Пример использования

In [ ]:
# ============================================
# 🚀 ПРИМЕР ЗАПУСКА ОЦЕНКИ
# ============================================

# Путь к датасету
DATASET_PATH = 'dataset'  # Измените на ваш путь

# Конфигурация моделей (укажите пути к сохраненным моделям)
MODELS_CONFIG = {
    '1D_CNN': {
        'type': 'pytorch',
        'loader': load_1d_cnn_model,
        'model_path': None,  # Например: './models/1d_cnn_best.pth'
        'input_channels': 8,
        'n_classes': 2
    },
    'CNN_LSTM': {
        'type': 'pytorch',
        'loader': load_cnn_lstm_model,
        'model_path': None,  # Например: './models/cnn_lstm_best.pth'
        'input_channels': 8,
        'n_classes': 2,
        'hidden_size': 128
    },
    'PipeTransformer': {
        'type': 'pytorch',
        'loader': load_transformer_model,
        'model_path': None,  # Например: './models/transformer_best.pth'
        'input_channels': 8,
        'n_classes': 2
    },
    '1D_CNN_XGBoost': {
        'type': 'xgboost',
        'loader': load_xgboost_model,
        'model_path': None  # Например: './models/xgboost_model.json'
    }
}

# Параметры оценки
BATCH_SIZE = 32
APPLY_AUGMENTATION = False  # Установить True для применения аугментации
N_AUGMENTED = 1  # Количество аугментированных копий на образец
RANDOM_STATE = 42
SAVE_RESULTS = True
OUTPUT_DIR = './evaluation_results'

# Запуск оценки (раскомментировать для запуска)
# results, data_dict = run_full_evaluation(
#     dataset_path=DATASET_PATH,
#     models_config=MODELS_CONFIG,
#     batch_size=BATCH_SIZE,
#     apply_augmentation=APPLY_AUGMENTATION,
#     n_augmented=N_AUGMENTED,
#     random_state=RANDOM_STATE,
#     save_results=SAVE_RESULTS,
#     output_dir=OUTPUT_DIR
# )

## 9. Отдельный запуск подготовки данных

In [ ]:
# ============================================
# 🔄 ТОЛЬКО ПОДГОТОВКА ДАННЫХ (без оценки моделей)
# ============================================

DATASET_PATH = 'dataset'  # Измените на ваш путь
APPLY_AUGMENTATION = False
N_AUGMENTED = 1
RANDOM_STATE = 42

# Загрузка данных
data_list, labels, filenames, thicknoms = load_dataset_from_folder(DATASET_PATH)

# Подготовка и разделение
data_dict = prepare_and_split_data(
    data_list, labels, filenames, thicknoms,
    train_size=0.7, val_size=0.15, test_size=0.15,
    apply_augmentation=APPLY_AUGMENTATION,
    n_augmented=N_AUGMENTED,
    random_state=RANDOM_STATE,
    save_indices=True,
    output_dir='./test_indices'
)

# Создание DataLoader
train_loader, val_loader, test_loader = create_dataloaders(data_dict, batch_size=32)

print("\n✅ Данные готовы для использования!")
print(f"   Train: {len(data_dict['X_train'])} образцов")
print(f"   Val: {len(data_dict['X_val'])} образцов")
print(f"   Test: {len(data_dict['X_test'])} образцов")